# Notebook: Building the PubMed Text Embedding and Retrieval Index

This notebook prepares the semantic text-retrieval module for our multimodal radiology agent. It builds the **PubMed abstract index** that powers evidence-grounded citation retrieval via FAISS and SPECTER2. This module enables the agent to return relevant biomedical literature when answering user queries or interpreting X-ray captions.

---
## Background

In order to support high-quality evidence generation, our multimodal radiology assistant includes a **text-retrieval module** that can return semantically relevant biomedical literature in response to user queries or image-derived captions. This module enhances factual grounding, citation transparency, and interpretability — all critical for clinical-facing AI systems.

We use the **`MedRAG/pubmed`** dataset hosted on Hugging Face, a cleaned subset of the PubMed Open Access collection. It contains over 2.2 million biomedical papers, each with:

- A stable `PMID` identifier
- A structured `title`
- An abstract stored in the `content` field

These papers span a wide range of clinical and research topics, making them ideal for embedding-based semantic search.

In this notebook, we sample a 200,000-record subset using keyword filtering and build a **FAISS-based text retrieval index** using **SPECTER2**, a state-of-the-art transformer model trained to embed scientific documents. This index enables fast, scalable retrieval of relevant PubMed papers based on either a user query or an automatically generated caption from a medical image.

This forms the **text tower** of our dual-retrieval system and powers the citation-generating component of our multi-agent architecture.

---
## Workflow Overview

### **Step 1 – Environment Setup**
- Verified GPU availability (T4 or A100) and activated device-aware logic.
- Installed required libraries:
  - `sentence-transformers` for SPECTER2 embedding
  - `datasets` for Hugging Face dataset loading
  - `faiss-cpu` for building a fast similarity index

### **Step 2 – Sample and Export PubMed Subset**
- Loaded the `MedRAG/pubmed` dataset from Hugging Face (2.2M entries).
- Sampled **200,000 records** using keyword filtering for prototype-scale text retrieval.
- Exported two files to `data/pubmed/`:
  - `raw_abstracts.jsonl` – Full records with `pmid`, `title`, `abstract`
  - `text_metadata.json` – Lightweight manifest for runtime lookup

### **Step 3 – Embed with SPECTER2**
- Loaded the `allenai/specter2_base` model (768-D output).
- Concatenated `title + abstract` → embedded in batches of 64 with normalization.
- Saved full embedding matrix as:
  - `text_vectors.npy` (float32, 30k × 768)

### **Step 4 – Build and Save FAISS Index**
- Created a cosine similarity FAISS index using `faiss.IndexFlatIP(dim=768)`.
- Added all SPECTER2 embeddings to the index.
- Persisted the index to disk as:
  - `text_faiss.bin` (≈88 MB)
  - Ensures O(1) semantic lookup during inference

### **Step 5 – Sanity Check: Query the Index**
- Queried the index with two test prompts:
  - `"chest x-ray findings in pulmonary embolism"`
  - `"case reports involving lung nodules in pediatric patients"`
- Verified that returned titles were semantically aligned and non-random.
- Confirmed full end-to-end retrieval stack is functional and reproducible.

---

### **Final Output Directory**
All artifacts are saved to the following directory for downstream use in the RAG pipeline:

```
data/pubmed/
├── raw_abstracts.jsonl       # Full 30k sample
├── text_metadata.json        # Lightweight manifest
├── text_vectors.npy          # 768-D SPECTER2 embeddings
└── text_faiss.bin            # FAISS cosine index (768-D)
```

This module completes the text-retrieval half of our citation-augmented agent. The next phase is MIMIC-QA curation and BioGPT-LoRA fine-tuning.

##Step 0: Mounting Google Drive and Importing Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/multimodal-xray-agent
!ls

In [ ]:
!pip install faiss-cpu -q

In [ ]:
!pip install --upgrade datasets huggingface_hub fsspec -q

In [4]:
import torch
import faiss
import random
import requests
import os, json
import numpy as np

from tqdm import tqdm
from datasets import load_dataset, load_from_disk
from sentence_transformers import SentenceTransformer

from src.text_search import query_text_faiss

## Step 1: Verifying GPU and Environment

In [5]:
# Device-agnostic setup
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    device = torch.device("cuda")
    print(f"GPU detected: {device_name}")
else:
    device = torch.device("cpu")
    print("GPU not detected. Falling back to CPU.")

print(f"Running on device: {device}")

GPU detected: NVIDIA A100-SXM4-40GB
Running on device: cuda


## Step 2: Sampling and Exporting PubMed Abstracts for Semantic Indexing

- Selected a representative subset of **30,000 abstracts** from the `MedRAG/pubmed` dataset on Hugging Face. This subset includes:
  - `title`: paper title
  - `abstract`: main summary text
  - `id`: PubMed ID (PMID)

- Defined output paths and directory structure:
  - `data/pubmed/raw_abstracts.jsonl`: line-delimited JSON containing full `title + abstract + pmid` entries.
  - `data/pubmed/text_metadata.json`: lightweight metadata file containing only `pmid` and `title` for retrieval post-indexing.

- Exported both files by iterating over the sampled records:
  - Trimmed whitespace and combined relevant fields.
  - Ensured all JSON was UTF-8 encoded and human-readable (indent=2 for metadata).

- These files serve as the **input corpus for downstream SPECTER2 embedding and FAISS indexing**.

In [6]:
# Define destination path
PUBMED_DIR = "data/pubmed_filtered"
os.makedirs(PUBMED_DIR, exist_ok=True)

In [7]:
# Define output paths
RAW_JSONL_PATH = os.path.join(PUBMED_DIR, "raw_abstracts.jsonl")
META_JSON_PATH = os.path.join(PUBMED_DIR, "text_metadata.json")
FAISS_INDEX_PATH = "data/pubmed/text_faiss.bin"
EMBEDDINGS_NPY_PATH = "data/pubmed/text_vectors.npy"

In [ ]:
# Downloading the data
dataset = load_dataset("MedRAG/pubmed", split="train")

In [9]:
len(dataset)

23898701

In [11]:
keywords = [
    # General radiology
    "radiology", "radiograph", "diagnostic imaging",

    # Modalities
    "x-ray", "chest x-ray", "cxr",

    # Anatomy - thoracic focus
    "chest", "thorax", "lungs", "pulmonary", "pleura", "diaphragm", "mediastinum",
    "heart", "cardiac", "rib", "clavicle", "scapula",

    # Common radiological findings
    "consolidation", "effusion", "atelectasis", "opacity", "mass", "nodule",
    "pneumothorax", "infiltrate", "interstitial", "hyperinflation", "emphysema",

    # Diseases of interest
    "copd", "asthma", "pneumonia", "tuberculosis", "lung cancer",
    "covid-19", "pulmonary fibrosis", "sarcoidosis", "pulmonary edema",
    "pulmonary hypertension",

    # Keywords to capture RAG-friendly abstracts
    "radiographic", "findings", "interpretation", "diagnosis", "chest imaging",
]

In [12]:
# Filter based on title + abstract
def is_radiology_related(example):
    text = (example["title"] + " " + example["content"]).lower()
    return any(kw in text for kw in keywords)

In [ ]:
filtered_ds = dataset.filter(is_radiology_related)

In [14]:
len(filtered_ds)

11250965

In [ ]:
filtered_ds.save_to_disk("/content/drive/MyDrive/multimodal-xray-agent/data/pubmed_filtered")

In [19]:
keywords = [
    "chest radiograph", "chest x-ray", "cxr", "pulmonary",
    "chest imaging", "thoracic imaging", "pleura", "mediastinum",
    "pulmonary nodule", "pulmonary opacity", "airspace disease",
    "pleural effusion", "pneumothorax", "atelectasis", "emphysema", "bibasilar",
    "interstitial lung disease", "copd", "cardiomegaly", "subpulmonic effusion",
    "hilar enlargement", "hyperinflation", "pulmonary hypertension", "interstitial"
]

In [ ]:
filtered_ds_2 = filtered_ds.filter(is_radiology_related)

In [21]:
len(filtered_ds_2)

762042

In [22]:
# Saving the dataset
SAVE_DIR = "data/pubmed_filtered/filtered_dataset_final"
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
filtered_ds_2.save_to_disk(SAVE_DIR)

In [25]:
filtered_ds_2 = filtered_ds_2.shuffle(seed=42)

In [26]:
# Select first 200,000 samples
filtered_subset = filtered_ds_2.select(range(200_000))

In [27]:
len(filtered_subset)

200000

In [28]:
SAVE_DIR = "data/pubmed_filtered/filtered_dataset_final_200k"
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
filtered_subset.save_to_disk(SAVE_DIR)

In [8]:
dataset = load_from_disk("data/pubmed_filtered/filtered_dataset_final_200k")

In [9]:
len(dataset)

200000

This code block performs two main tasks: writing the sampled PubMed records to a `.jsonl` file and extracting minimal metadata for quick lookup.

- Opens two output files:
  - `raw_abstracts.jsonl` (line-delimited JSON): stores full records (PMID, title, abstract)
  - `text_meta.json`: stores only the `pmid` and `title` for lightweight indexing

- Iterates through each entry in the dataset:
  - Strips whitespace from `title` and `abstract`
  - Writes a JSON line to `raw_abstracts.jsonl` with all key fields
  - Appends a compact metadata dict (`pmid`, `title`) to an in-memory list

- After the loop:
  - Dumps the metadata list to `text_meta.json` using `json.dump(...)`

In [11]:
# Write abstracts to JSONL and minimal metadata to JSON
with open(RAW_JSONL_PATH, "w", encoding="utf-8") as f_jsonl, \
     open(META_JSON_PATH, "w", encoding="utf-8") as f_meta:

    metadata = []
    for entry in dataset:
        title = entry["title"].strip()
        abstract = entry["content"].strip()
        pmid = entry["id"]

        # Save full abstract line
        json_line = json.dumps({
            "pmid": pmid,
            "title": title,
            "abstract": abstract
        })
        f_jsonl.write(json_line + "\n")

        # Save metadata for quick lookup
        metadata.append({"pmid": pmid, "title": title})

    json.dump(metadata, f_meta, indent=2)

print(f"Saved: {RAW_JSONL_PATH}")
print(f"Saved: {META_JSON_PATH}")

Saved: data/pubmed_filtered/raw_abstracts.jsonl
Saved: data/pubmed_filtered/text_metadata.json


## Step 3: Embedding Title + Abstracts using SPECTER2

- Loaded the `raw_abstracts.jsonl` file line-by-line and concatenated each paper's `title` and `abstract` into a single string.
- Used `allenai/specter2_base`, a state-of-the-art transformer model trained for scientific document similarity, via `SentenceTransformer`.
- Batched the input texts and encoded them into dense vector representations:
  - Batch size = 64
  - Output = normalized embeddings (`float32`, shape: `[N, 768]`)
- Persisted the resulting matrix to disk as `pubmed_embeddings.npy` for reproducibility and future use.

In [12]:
# Load abstracts
print("Reading abstracts for embedding...")
texts = []
with open(RAW_JSONL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        text = f"{record['title']} {record['abstract']}"
        texts.append(text.strip())

Reading abstracts for embedding...


In [ ]:
# Load SPECTER2 encoder
model = SentenceTransformer("allenai/specter2_base")

In [ ]:
# Generate embeddings
embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

# Save embeddings to disk (optional)
np.save(EMBEDDINGS_NPY_PATH, embeddings)

## Step 4 – Build and Save FAISS Index

- Constructed a dense vector index using **FAISS** with inner product (cosine similarity on normalized vectors).
- FAISS index type: `IndexFlatIP(dim=768)` – a flat (brute-force) index suitable for exact nearest neighbor retrieval.
- Added all `SPECTER2` embeddings to the index as `float32` numpy arrays.
- Saved the resulting binary index to disk as `text_faiss.bin` for fast semantic retrieval at inference time.

In [15]:
# Build FAISS index (cosine similarity)
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(np.asarray(embeddings, dtype=np.float32))
faiss.write_index(index, FAISS_INDEX_PATH)

print(f"FAISS index saved to: {FAISS_INDEX_PATH}")
print(f"Embeddings saved to: {EMBEDDINGS_NPY_PATH}")

FAISS index saved to: data/pubmed/text_faiss.bin
Embeddings saved to: data/pubmed/text_vectors.npy


## Step 5: Sanity Check (Query Text Index)

- Loaded the saved `FAISS` index (`text_faiss.bin`) and its corresponding metadata (`text_meta.json`) into memory.
- Defined a simple semantic search function `query_text_faiss()` that:
  - Encodes an input query using the same `SPECTER2` model.
  - Performs top-`k` nearest neighbor search using the FAISS index.
  - Returns the most semantically similar papers based on cosine similarity scores.
- Ran two test queries to verify semantic alignment:
  1. `"chest x-ray findings in pulmonary embolism"`
  2. `"case reports involving lung nodules in pediatric patients"`
- Validated that top-ranked papers had relevant titles, confirming embedding + indexing pipeline is working correctly.

In [16]:
# Load FAISS index and metadata
index = faiss.read_index(FAISS_INDEX_PATH)

with open(META_JSON_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

In [17]:
model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)

In [19]:
query = (
    "Frontal chest radiograph shows bilateral perihilar opacities consistent with pulmonary edema. "
    "Cardiomegaly is present. No evidence of pneumothorax or pleural effusion."
)

hits = query_text_faiss(query, model, index, metadata, top_k=5)

print(f"\n Query: {query}\n")
for i, hit in enumerate(hits, 1):
    print(f"{i}. [{hit['score']:.3f}] {hit['title']} (PMID: {hit['pmid']})")


 Query: Frontal chest radiograph shows bilateral perihilar opacities consistent with pulmonary edema. Cardiomegaly is present. No evidence of pneumothorax or pleural effusion.

1. [0.906] A case of pneumonia in a critical patient. (PMID: pubmed23n0522_12759)
2. [0.902] [Dyspnea, fatigue and subpulmonary mass]. (PMID: pubmed23n0730_24833)
3. [0.902] A Rare Case of Bronchial Anthracofibrosis with Pleural Anthracosis Presented as Recurrent Pleural Effusion. (PMID: pubmed23n1005_14369)
4. [0.901] Radiographical Spectrum of High-altitude Pulmonary Edema: A Pictorial Essay. (PMID: pubmed23n1095_10069)
5. [0.901] [Para-septal emphysema]. (PMID: pubmed23n0068_596)


In [20]:
query = (
    "Chest x-ray reveals right-sided pleural effusion with associated lung base opacity. "
    "Heart size within normal limits. No pneumothorax observed."
)

hits = query_text_faiss(query, model, index, metadata, top_k=5)

print(f"\n Query: {query}\n")
for i, hit in enumerate(hits, 1):
    print(f"{i}. [{hit['score']:.3f}] {hit['title']} (PMID: {hit['pmid']})")


 Query: Chest x-ray reveals right-sided pleural effusion with associated lung base opacity. Heart size within normal limits. No pneumothorax observed.

1. [0.930] A Rare Case of Bronchial Anthracofibrosis with Pleural Anthracosis Presented as Recurrent Pleural Effusion. (PMID: pubmed23n1005_14369)
2. [0.918] [Pneumothorax Secondary to Pulmonary Tuberculosis]. (PMID: pubmed23n0925_17870)
3. [0.915] Epidermoid cyst radiologically mistaken as a left sided subpulmonic effusion. (PMID: pubmed23n0573_10556)
4. [0.914] Electrocardiographic manifestations in a large right-sided pneumothorax. (PMID: pubmed23n1077_3628)
5. [0.913] [Dyspnea, fatigue and subpulmonary mass]. (PMID: pubmed23n0730_24833)


In [21]:
query = (
    "There is hyperinflation of both lungs with flattened diaphragms. "
    "Increased lucency in bilateral lung fields suggests underlying emphysema."
)

hits = query_text_faiss(query, model, index, metadata, top_k=5)

print(f"\n Query: {query}\n")
for i, hit in enumerate(hits, 1):
    print(f"{i}. [{hit['score']:.3f}] {hit['title']} (PMID: {hit['pmid']})")


 Query: There is hyperinflation of both lungs with flattened diaphragms. Increased lucency in bilateral lung fields suggests underlying emphysema.

1. [0.888] An unusual case of unilateral hyperlucent lung. (PMID: pubmed23n1074_2171)
2. [0.883] Lung Hyperlucency: A Clinical-Radiologic Algorithmic Approach to Diagnosis. (PMID: pubmed23n0998_24397)
3. [0.882] Upper lobe-dominant pulmonary fibrosis showing deposits of hard metal component in the fibrotic lesions. (PMID: pubmed23n0674_7981)
4. [0.880] Management of congenital lobar overinflation in an adult. (PMID: pubmed23n1042_10834)
5. [0.874] Challenges in pulmonary fibrosis: 6--Problematic granulomatous lung disease. (PMID: pubmed23n0588_2102)


In [22]:
query = (
    "Frontal chest radiograph demonstrates clear lung fields with no evidence of focal consolidation, effusion, or pneumothorax."
    "Cardiac silhouette and mediastinal contours are within normal limits. No acute osseous abnormalities."
)

hits = query_text_faiss(query, model, index, metadata, top_k=5)

print(f"\n Query: {query}\n")
for i, hit in enumerate(hits, 1):
    print(f"{i}. [{hit['score']:.3f}] {hit['title']} (PMID: {hit['pmid']})")


 Query: Frontal chest radiograph demonstrates clear lung fields with no evidence of focal consolidation, effusion, or pneumothorax.Cardiac silhouette and mediastinal contours are within normal limits. No acute osseous abnormalities.

1. [0.898] [Thoracic pathology without apparent radiographic sign: re-interpretation basis]. (PMID: pubmed23n0382_23904)
2. [0.897] Pneumothorax in the Supine Patient: Subtle Radiographic Signs. (PMID: pubmed23n0865_14045)
3. [0.897] Ultrasound of the thorax (noncardiac). (PMID: pubmed23n0631_17308)
4. [0.896] [Dyspnea, fatigue and subpulmonary mass]. (PMID: pubmed23n0730_24833)
5. [0.896] A Rare Case of Bronchial Anthracofibrosis with Pleural Anthracosis Presented as Recurrent Pleural Effusion. (PMID: pubmed23n1005_14369)
